In [1]:
!pip install mujoco brax datasets flax

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 17.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import brax
import jax
import mujoco
from mujoco import mjx

print(f"Brax version: {brax.__version__}")
print(f"JAX version: {jax.__version__}")
print(f"MuJoCo version: {mujoco.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"JAX backend: {jax.default_backend()}")

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
Brax version: 0.14.2
JAX version: 0.9.2
MuJoCo version: 3.6.0


E0000 00:00:1775633021.360494      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
JAX backend: tpu


In [3]:
"""
UAV Sim2Real Training Demo with MuJoCo XLA (MJX), JAX, Brax using OpenSource Dataset from Hugging Face.
Optimized drone reinforcement learning training script for Kaggle TPU v5e-8.
"""

import os
import warnings

# Suppress Brax deprecation warnings (since we are correctly using MJX backend)
warnings.filterwarnings("ignore", category=UserWarning)

# --- SAFE MEMORY ALLOCATION ---
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9"

import jax
from jax import numpy as jnp
from jax.sharding import Mesh, PartitionSpec, NamedSharding
from jax.experimental import mesh_utils
import mujoco
from mujoco import mjx

# Use PipelineEnv (required by Brax 0.14.x)
from brax.envs.base import State, PipelineEnv
from brax.io import mjcf as brax_mjcf
from brax.training.agents.ppo import train as ppo
from brax.io import model as brax_model
from brax.io import html

import pandas as pd
import numpy as np
import ast
import time
import glob
from huggingface_hub import HfApi, snapshot_download

# ==========================================
# 1. Data Processing and Batch Loading Logic
# ==========================================
def parse_dataframe(df, num_points):
    """Helper function: Parse coordinate points from DataFrames with different structures"""
    df = df.head(num_points)
    cols = df.columns.tolist()

    if 'x' in cols and 'y' in cols and 'z' in cols:
        return np.column_stack((df['x'], df['y'], df['z']))
    elif 'tx' in cols and 'ty' in cols and 'tz' in cols:
        return np.column_stack((df['tx'], df['ty'], df['tz']))
    elif 'position' in cols:
        positions = df['position'].tolist()
        if isinstance(positions[0], str):
            positions = [ast.literal_eval(p) for p in positions]
        return np.array(positions)
    else:
        x_col = next((c for c in cols if c.lower() in ['x', 'tx'] or c.lower().endswith('.x') or c.lower().endswith('_x')), None)
        y_col = next((c for c in cols if c.lower() in ['y', 'ty'] or c.lower().endswith('.y') or c.lower().endswith('_y')), None)
        z_col = next((c for c in cols if c.lower() in ['z', 'tz'] or c.lower().endswith('.z') or c.lower().endswith('_z')), None)
        if x_col and y_col and z_col:
            return np.column_stack((df[x_col], df[y_col], df[z_col]))
        else:
            raise ValueError(f"Unrecognized column names. Available columns are: {cols}")

def get_csv_file_lists(repo_id, hf_token=None):
    """Get all CSV filenames in the repo and split them equally into two batches"""
    print(f"Fetching full file list from {repo_id}...")
    api = HfApi(token=hf_token)
    all_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
    csv_files = [f for f in all_files if f.endswith('.csv')]
    mid_point = len(csv_files) // 2
    batch1 = csv_files[:mid_point]
    batch2 = csv_files[mid_point:]
    print(f"Discovered a total of {len(csv_files)} CSV files.")
    print(f"Split into two batches: Batch 1 ({len(batch1)} files), Batch 2 ({len(batch2)} files).")
    return batch1, batch2

def download_batch_and_extract_demo(repo_id, file_list, hf_token, num_demo_points=4):
    """
    Download specified batch of files (multi-threaded), and extract a certain number of waypoints for Demo training.
    """
    print(f"\nStarting batch download ({len(file_list)} files in total)...")
    local_dir = snapshot_download(
        repo_id=repo_id,
        repo_type="dataset",
        allow_patterns=file_list,
        token=hf_token,
        max_workers=8
    )
    local_csvs = glob.glob(os.path.join(local_dir, "**/*.csv"), recursive=True)
    local_csvs.sort()
    if not local_csvs:
        raise FileNotFoundError("Could not find downloaded CSV files locally!")
    print(f"Batch download complete! {len(local_csvs)} related files exist locally.")
    print(f"Extracting {num_demo_points} trajectory points from the first file ({os.path.basename(local_csvs[0])})...")
    df = pd.read_csv(local_csvs[0])
    waypoints = parse_dataframe(df, num_points=num_demo_points)
    print(f"Successfully extracted {len(waypoints)} waypoints!")
    return jnp.array(waypoints, dtype=jnp.float32)

# ==========================================
# 2. MuJoCo UAV (Quadrotor) Model Definition (XML)
# ==========================================
# THE ULTIMATE OOM FIX: Changed integrator from "RK4" to "Euler". 
# RK4 evaluates physics derivatives 4 times per step, multiplying the XLA compilation graph size by 4x,
# causing the 330GB+ Host RAM explosion. Euler evaluates once, dropping RAM usage by 75%+ during compilation.
UAV_XML = """
<mujoco model="quadrotor">
  <compiler angle="degree" inertiafromgeom="true"/>
  <option gravity="0 0 -9.81" timestep="0.01" integrator="Euler"/>

  <default>
    <geom friction="1 0.1 0.1" margin="0.001" rgba="0.8 0.6 0.4 1"/>
    <joint damping="0.01"/>
  </default>

  <worldbody>
    <light pos="0 0 10" dir="0 0 -1" diffuse="1 1 1"/>
    <geom type="plane" size="10 10 0.1" rgba="0.9 0.9 0.9 1"/>

    <!-- UAV Body -->
    <body name="uav" pos="0 0 1">
      <freejoint name="root"/>
      <geom name="core" type="box" size="0.1 0.1 0.05" mass="1.0" rgba="0.2 0.5 0.8 1"/>
      <site name="rotor1" pos="0.1 0.1 0" size="0.05" rgba="1 0 0 1"/>
      <site name="rotor2" pos="-0.1 0.1 0" size="0.05" rgba="1 0 0 1"/>
      <site name="rotor3" pos="-0.1 -0.1 0" size="0.05" rgba="0 1 0 1"/>
      <site name="rotor4" pos="0.1 -0.1 0" size="0.05" rgba="0 1 0 1"/>
    </body>
  </worldbody>

  <actuator>
    <motor name="m1" site="rotor1" ctrlrange="0 5" ctrllimited="true"/>
    <motor name="m2" site="rotor2" ctrlrange="0 5" ctrllimited="true"/>
    <motor name="m3" site="rotor3" ctrlrange="0 5" ctrllimited="true"/>
    <motor name="m4" site="rotor4" ctrlrange="0 5" ctrllimited="true"/>
  </actuator>
</mujoco>
"""

# ==========================================
# 3. Brax/MJX RL Environment
# ==========================================
class UAVTrackingEnv(PipelineEnv):
    def __init__(self, waypoints):
        sys = brax_mjcf.loads(UAV_XML)
        super().__init__(sys=sys, backend='mjx', n_frames=1)
        self.waypoints = waypoints
        self.num_waypoints = waypoints.shape[0]

    def reset(self, rng: jnp.ndarray) -> State:
        rng, rng_state = jax.random.split(rng)
        
        # FIX: Spawn UAV near the first waypoint instead of (0,0,1)
        target_pos = self.waypoints[0]
        # 1.0m to Z axis to ensure it spawns safely above ground
        init_pos = target_pos + jnp.array([0.0, 0.0, 1.0])
        init_q = self.sys.init_q.at[:3].set(init_pos)
        
        pipeline_state = self.pipeline_init(
            init_q + jax.random.uniform(rng_state, (self.sys.q_size(),), minval=-0.1, maxval=0.1),
            jnp.zeros(self.sys.qd_size())
        )
        obs = self._get_obs(pipeline_state, target_idx=jnp.array(0))
        uav_pos = pipeline_state.q[:3]
        target_pos = self.waypoints[0]
        initial_distance = jnp.linalg.norm(uav_pos - target_pos)
        return State(
            pipeline_state=pipeline_state,
            obs=obs,
            reward=jnp.array(0.0),
            done=jnp.array(0.0),
            metrics={"target_idx": jnp.array(0.0), "distance_to_target": initial_distance}
        )

    def step(self, state: State, action: jnp.ndarray) -> State:
        pipeline_state = self.pipeline_step(state.pipeline_state, action)
        target_idx = state.metrics["target_idx"].astype(jnp.int32)

        uav_pos = pipeline_state.q[:3]
        target_pos = self.waypoints[target_idx]
        distance = jnp.linalg.norm(uav_pos - target_pos)

        reward = -distance

        reached = distance < 0.2
        next_idx = jnp.minimum(target_idx + 1, self.num_waypoints - 1)
        target_idx = jnp.where(reached, next_idx, target_idx).astype(jnp.float32)

        done = jnp.where(uav_pos[2] < 0.1, 1.0, 0.0)
        done = jnp.where(distance > 20.0, 1.0, done)

        obs = self._get_obs(pipeline_state, target_idx.astype(jnp.int32))
        
        metrics = state.metrics.copy()
        metrics["target_idx"] = target_idx
        metrics["distance_to_target"] = distance
        
        return state.replace(pipeline_state=pipeline_state, obs=obs, reward=reward, done=done, metrics=metrics)

    def _get_obs(self, pipeline_state, target_idx) -> jnp.ndarray:
        uav_pos  = pipeline_state.q[:3]
        uav_quat = pipeline_state.q[3:7]
        uav_vel  = pipeline_state.qd[:3]
        target_pos = self.waypoints[target_idx]
        return jnp.concatenate([uav_pos, uav_quat, uav_vel, target_pos])

    @property
    def action_size(self) -> int: return 4
    @property
    def observation_size(self) -> int: return 13

## Train

In [4]:
print(f"JAX Devices: {jax.device_count()} (TPU cores expected: 8 on Kaggle)")

JAX Devices: 8 (TPU cores expected: 8 on Kaggle)


In [5]:
devices = mesh_utils.create_device_mesh((jax.device_count(),))
mesh = Mesh(devices, axis_names=('batch',))
print(f"JAX Hardware Devices: {jax.device_count()} TPU cores")
print(f"Mesh created: {mesh}")

JAX Hardware Devices: 8 TPU cores
Mesh created: Mesh('batch': 8, axis_types=(Auto,))


In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("HF_TOKEN")
hf_token = user_secrets.get_secret("HF_TOKEN_WRITE")

In [7]:
REPO_ID = "riotu-lab/Synthetic-UAV-Flight-Trajectories"

batch1_files, batch2_files = get_csv_file_lists(REPO_ID, hf_token)
stage1_start_time = time.time()

# --- STAGE 1 ---
print("\n[STAGE 1] ---------------------------------------------")
waypoints_batch1 = download_batch_and_extract_demo(REPO_ID, batch1_files, hf_token, num_demo_points=4)

print("\nStarting Stage 1 PPO training (Based on Batch 1 data)...")
env_1 = UAVTrackingEnv(waypoints_batch1)

# With Euler integrator, compilation memory is drastically reduced.
# We can now safely use a balanced batch configuration.
# Math check: num_envs(128) * unroll_length(10) = 1280 transitions
# batch_size(80) * num_minibatches(16) = 1280 transitions
with mesh:
    make_inference_fn_1, params_1, _ = ppo.train(
        environment=env_1,
        num_timesteps=10_000,
        num_evals=2,
        reward_scaling=1.0,
        episode_length=50,
        normalize_observations=False, 
        action_repeat=1,
        unroll_length=10,        
        num_minibatches=16,       
        num_updates_per_batch=4,
        discounting=0.99,
        learning_rate=3e-4,
        entropy_cost=1e-3,
        num_envs=128,            
        batch_size=80,           
        seed=42,
    )
print("Stage 1 training completed successfully.")

# --- Memory Cleanup between stages ---
print("\n[Memory Cleanup] Clearing JAX compilation cache...")
jax.clear_caches()
# Explicitly clear old env to free up CPU RAM
del env_1
import gc
gc.collect()
print("Cache cleared.")

# --- Forced Cooldown ---
print("\n[API Protection Mechanism] ------------------------------------------")
elapsed_time = time.time() - stage1_start_time
wait_target = 310
if elapsed_time < wait_target:
    sleep_duration = wait_target - elapsed_time
    print(f"Only {elapsed_time:.1f}s elapsed. Sleeping {sleep_duration:.1f}s to reset HF API quota...")
    time.sleep(sleep_duration)
    print("Wait over! API quota has been reset.")
else:
    print(f"Stage 1 took {elapsed_time:.1f}s, safely exceeding the 5-minute limit. Continuing directly!")

# --- STAGE 2 ---
print("\n[STAGE 2] ---------------------------------------------")
waypoints_batch2 = download_batch_and_extract_demo(REPO_ID, batch2_files, hf_token, num_demo_points=4)

print("\nStarting Stage 2 Continual Learning...")
env_2 = UAVTrackingEnv(waypoints_batch2)

with mesh:
    make_inference_fn_2, params_2, _ = ppo.train(
        environment=env_2,
        num_timesteps=10_000,
        num_evals=2,
        restore_params=params_1,  # Inherit weights from Stage 1
        reward_scaling=1.0,
        episode_length=50,
        normalize_observations=False, 
        action_repeat=1,
        unroll_length=10,
        num_minibatches=16,       
        num_updates_per_batch=4,
        discounting=0.99,
        learning_rate=3e-4,
        entropy_cost=1e-3,
        num_envs=128,
        batch_size=80,
        seed=99,
    )
print("Stage 2 continual training completed.")

Fetching full file list from riotu-lab/Synthetic-UAV-Flight-Trajectories...
Discovered a total of 5093 CSV files.
Split into two batches: Batch 1 (2546 files), Batch 2 (2547 files).

[STAGE 1] ---------------------------------------------

Starting batch download (2546 files in total)...


Fetching 2546 files:   0%|          | 0/2546 [00:00<?, ?it/s]

gazebo_trajectory2D-2_100.csv: 0.00B [00:00, ?B/s]

gazebo_trajectory2D-2_1001.csv: 0.00B [00:00, ?B/s]

gazebo_trajectory2D-2_1006.csv: 0.00B [00:00, ?B/s]

...

gazebo_trajectory_96.csv: 0.00B [00:00, ?B/s]

gazebo_trajectory_98.csv: 0.00B [00:00, ?B/s]

gazebo_trajectory_99.csv: 0.00B [00:00, ?B/s]

gazebo_trajectory_97.csv: 0.00B [00:00, ?B/s]

Batch download complete! 5093 related files exist locally.
Extracting 4 trajectory points from the first file (gazebo_trajectory2D-2_100.csv)...
Successfully extracted 4 waypoints!

Starting Stage 2 Continual Learning...
Stage 2 continual training completed.


## Save

In [8]:
# --- STAGE 3: Save model ---
print("\n[Wrap Up] ---------------------------------------------")
model_path = "uav_continual_ppo_policy_8samples.pkl"
brax_model.save_params(model_path, params_2)
print(f"Final model saved to: '{model_path}'")


[Wrap Up] ---------------------------------------------
Final model saved to: 'uav_continual_ppo_policy_8samples.pkl'


## Inference

In [13]:
# --- STAGE 4: Inference ---
print("\nStarting UAV flight test on Batch 2 trajectories...")
loaded_params = brax_model.load_params(model_path)

# --- STANDALONE INFERENCE RECONSTRUCTION ---
# Import network builder tools strictly for standalone execution
from brax.training.agents.ppo import networks as ppo_networks
from brax.training.agents.ppo import train as ppo_train

# 1. Rebuild the exact network architecture used during training
ppo_network = ppo_networks.make_ppo_networks(
    env_2.observation_size,
    env_2.action_size,
    preprocess_observations_fn=lambda x, y: x  # Must match normalize_observations=False
)

# 2. Re-create the inference generator and bind the loaded parameters
# FIX: make_inference_fn is located in ppo_networks, not ppo_train
standalone_inference_generator = ppo_networks.make_inference_fn(ppo_network)
policy_fn = standalone_inference_generator(loaded_params)
# -------------------------------------------

jit_reset  = jax.jit(env_2.reset)
jit_step   = jax.jit(env_2.step)
jit_policy = jax.jit(policy_fn)

rng = jax.random.PRNGKey(123)
rng, rng_reset = jax.random.split(rng)
state = jit_reset(rng_reset)
print("UAV Takeoff!")

# List to store trajectory states for visualization
rollout = [state.pipeline_state]

for step in range(150):
    rng, rng_act = jax.random.split(rng)
    ctrl, _ = jit_policy(state.obs, rng_act)
    state = jit_step(state, ctrl)
    
    # Save state for rendering
    rollout.append(state.pipeline_state)

    if step % 10 == 0:
        dist = state.metrics['distance_to_target']
        target_idx = state.metrics['target_idx']
        print(f"Time Step {step:03d} | Tracking Waypoint {int(target_idx)} | Distance Error: {dist:.3f} m")

    if state.done:
        print(f"UAV crashed or flew out of bounds. Episode terminated early at step {step}.")
        break

print("All processes demonstrated successfully!")

# Generate HTML Visualization
print("\n[Visualization] Generating 3D flight trajectory HTML...")
html_content = html.render(env_2.sys.tree_replace({'opt.timestep': env_2.dt}), rollout)
with open("uav_flight_trajectory.html", "w") as f:
    f.write(html_content)
print("Visualization saved to 'uav_flight_trajectory.html'. Download it from Kaggle outputs to view the flight!")


Starting UAV flight test on Batch 2 trajectories...
UAV Takeoff!
Time Step 000 | Tracking Waypoint 0 | Distance Error: 0.932 m
Time Step 010 | Tracking Waypoint 0 | Distance Error: 0.869 m
Time Step 020 | Tracking Waypoint 0 | Distance Error: 0.708 m
Time Step 030 | Tracking Waypoint 0 | Distance Error: 0.451 m
Time Step 040 | Tracking Waypoint 3 | Distance Error: 0.194 m
Time Step 050 | Tracking Waypoint 3 | Distance Error: 0.560 m
Time Step 060 | Tracking Waypoint 3 | Distance Error: 1.086 m
Time Step 070 | Tracking Waypoint 3 | Distance Error: 1.729 m
Time Step 080 | Tracking Waypoint 3 | Distance Error: 2.474 m
Time Step 090 | Tracking Waypoint 3 | Distance Error: 3.320 m
Time Step 100 | Tracking Waypoint 3 | Distance Error: 4.264 m
Time Step 110 | Tracking Waypoint 3 | Distance Error: 5.307 m
Time Step 120 | Tracking Waypoint 3 | Distance Error: 6.447 m
Time Step 130 | Tracking Waypoint 3 | Distance Error: 7.686 m
Time Step 140 | Tracking Waypoint 3 | Distance Error: 9.023 m
All 